# S04 · Stacks, Queues, BSTs & Symbol Tables
### Guided, ungraded practical · Python 3 / Google Colab
**Predict → run → modify → explain.** No installation required. This is practice, not a graded assignment.

## Objectives and rules
1. Use a stack to match nested brackets (LIFO).
2. Use a queue to dispatch tasks in arrival order (FIFO).
3. Build a BST-backed symbol table with unique comparable keys.
4. Explain deletion, traversal, and tree-height complexity.

**Challenge points:** 2 for a correct prediction; 2 for an edge-case explanation; 2 for naming an invariant. Record points on paper. Hints do not affect any formal grade.

## Step 1 — Stack predictions
Predict the popped value after pushing A, B, C. Write your answer before running.

In [ ]:
stack = []
for label in ["A", "B", "C"]:
    stack.append(label)
print("Stack before pop:", stack)
popped = stack.pop()
print("Popped:", popped, "| after:", stack)
assert popped == "C"

## Step 2 — Balanced delimiters
Trace `{a[(b+c)]}` and `([)]` character by character. What remains on the stack when a mismatch occurs?

In [ ]:
PAIRS = {")": "(", "]": "[", "}": "{"}
def balanced_delimiters(text):
    stack = []
    for ch in text:
        if ch in "([{":
            stack.append(ch)
        elif ch in PAIRS:
            if not stack or stack.pop() != PAIRS[ch]:
                return False
    return not stack

for sample in ["{a[(b+c)]}", "([)]", "((", "", "{}[]()"]:
    print(repr(sample), "->", balanced_delimiters(sample))
assert balanced_delimiters("{}[]()")
assert not balanced_delimiters("([)]")

**Try it:** Change `sample` to `}{`, then `[(])`. Explain the *prefix invariant*: unmatched opening brackets on the stack are exactly the open delimiters still awaiting a close.

## Step 3 — FIFO queue
Predict dispatch order for `Task A`, `Task B`, `Task C`. Python `deque.popleft()` removes at the front in O(1) time.

In [ ]:
from collections import deque
tasks = deque()
for name in ["Task A", "Task B", "Task C"]:
    tasks.append(name)
processed = []
while tasks:
    processed.append(tasks.popleft())
print("Processed:", processed)
assert processed == ["Task A", "Task B", "Task C"]

**Try it:** Add a fourth task and a task arriving after the first dequeue. Which task should leave next? What is inefficient about `list.pop(0)`?

## Step 4 — BST symbol table
The code below is a *guided demonstration* of a basic unbalanced BST. It is not a balanced tree: cost depends on height h.

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class Node:
    key: int
    value: str
    left: Optional["Node"] = None
    right: Optional["Node"] = None

class BSTMap:
    def __init__(self):
        self.root = None

    def put(self, key, value):
        def insert(node):
            if node is None:
                return Node(key, value)
            if key < node.key:
                node.left = insert(node.left)
            elif key > node.key:
                node.right = insert(node.right)
            else:
                node.value = value
            return node
        self.root = insert(self.root)

    def get(self, key):
        node = self.root
        while node is not None:
            if key < node.key:
                node = node.left
            elif key > node.key:
                node = node.right
            else:
                return node.value
        raise KeyError(key)

    def delete(self, key):
        def remove(node, target):
            if node is None:
                raise KeyError(target)
            if target < node.key:
                node.left = remove(node.left, target)
            elif target > node.key:
                node.right = remove(node.right, target)
            elif node.left is None:
                return node.right
            elif node.right is None:
                return node.left
            else:
                successor = node.right
                while successor.left is not None:
                    successor = successor.left
                node.key, node.value = successor.key, successor.value
                node.right = remove(node.right, successor.key)
            return node
        self.root = remove(self.root, key)

    def inorder(self):
        keys = []
        def visit(node):
            if node is not None:
                visit(node.left)
                keys.append(node.key)
                visit(node.right)
        visit(self.root)
        return keys

## Step 5 — Predict the tree
Insert 50, 30, 70, 20, 40, 60, 80. Before running, sketch the tree and the search path to 60.

In [ ]:
tree = BSTMap()
for key in [50, 30, 70, 20, 40, 60, 80]:
    tree.put(key, f"value-{key}")
print("In-order keys:", tree.inorder())
print("Value of 60:", tree.get(60))
assert tree.inorder() == [20, 30, 40, 50, 60, 70, 80]

## Step 6 — Trace deletion
**Predict** the new root after deleting 50. A node with two children is replaced by the smallest key in its right subtree (in-order successor).

In [ ]:
tree.delete(50)
print("Keys after deleting 50:", tree.inorder())
print("New root key:", tree.root.key)
assert tree.root.key == 60
assert tree.inorder() == [20, 30, 40, 60, 70, 80]

## Step 7 — Edge cases and self-check
Run the assertions below. Explain why tests alone cannot prove correctness for every input.

In [ ]:
empty = BSTMap()
assert empty.inorder() == []
single = BSTMap()
single.put(1, "one")
single.put(1, "updated")
assert single.inorder() == [1]
assert single.get(1) == "updated"
single.delete(1)
assert single.inorder() == []
try:
    single.get(123)
except KeyError:
    print("Missing key correctly raises KeyError")
else:
    raise AssertionError("Expected KeyError")
print("All notebook demonstration checks passed.")

## Step 8 — Your exploration (ungraded)
1. Build a fresh BST using keys `[1, 2, 3, 4, 5]`; sketch its shape. Compare with `[3, 1, 5, 2, 4]`.
2. Trace deleting a leaf, then a one-child node, then a two-child node from **fresh** trees.
3. Compare worst-case lookup costs for an unsorted array, a sorted array, a linked list and a plain BST. State the assumptions.
4. Write one correctness invariant for bracket matching and one for BST ordering.

**Exit ticket:** What is the difference between O(h) and O(log n) for an *unbalanced* BST?

**Next session:** S05 compares array/linked-list/BST symbol tables and prepares reproducible benchmarking. Official graded requirements remain governed by the institution.